# Day 16: Shared Memory Reduction

每个 block 将局部元素放 shared memory，树形归约每轮同步；越界线程贡献零。示例最后在 CPU 合并 block sums，不伪装成全 GPU 最优实现。

参考源码：`learning/kernels/vector_reduce.cu`。

每日概念、误区、面试问答：[知识讲义](../docs/KNOWLEDGE_DAY15_21.md)。

当前个人进度以[进度记录](../docs/LEARNING_PROGRESS.md)为准；运行参考答案不等于已独立掌握。

[Day 16完整讲义](../docs/DAY16_SHARED_REDUCTION.md)：概念、计算契约、源码实验、调试优化、答案与资料。

先写自己的预测，再执行原有代码单元格；不要自动勾选个人验收。

## 1. 图解与数字例子

8个数字求和可像折叠纸条：

```text
[1,2,3,4,5,6,7,8]
第1轮：前4项分别加后4项 → [6,8,10,12]
第2轮：前2项加后2项    → [16,20]
第3轮：第1项加第2项    → [36]
```

每轮后大家确认上一轮写完，再进入下一轮，这就是block barrier的用途。
N=257时，两个block先产生两个partial sums，再由CPU合并。
**本课没有把跨block的全局归约伪装成同一个GPU block里的同步。**

## 先预测

执行前写下预期 shape、输出或调度顺序；在下面另建 Markdown 单元格保存自己的答案，不修改参考实现作为唯一证据。

In [ ]:
import json
import os
import sys
from pathlib import Path
root = Path.cwd()
if root.name == 'learning':
    root = root.parent
if not (root / 'learning/curriculum.py').is_file():
    raise RuntimeError('Open from repository root or learning directory.')
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import torch
from learning.curriculum import run_day
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)


In [ ]:
report = run_day(16, DEVICE)
print(json.dumps(report, indent=2, ensure_ascii=False))
destination = Path('logs') / f"day{report['day']:02d}-course.json"
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(json.dumps(report, indent=2, ensure_ascii=False) + '\n')
print('Report:', destination)


## 自测与失败实验

为何 __syncthreads 不能只写在部分线程执行的条件里？

记录你改变的唯一变量、预期、实际结果和差异原因。

`requires_cuda` / `requires_cuda_compiler` / `external_service_not_configured` 都不是通过。

## 5. 自测与参考答案

**问：最后block只有一个有效元素，其他线程能直接return吗？**
答：本程序的同步模式下不应这样做；无效线程补0并参与协作。

**问：一个block算完就能读另一个block结果吗？**
答：没有隐式执行次序或全局barrier；要用有定义的跨kernel/全局协作方案。

**问：为什么CPU double与FP32 partial不要求逐位一样？**
答：运算精度和加法次序不同，应报告合理容差与输入。

**问：当前最终合并在哪？**
答：CPU，不是完整全GPU reduction。

完整实现限制、调试步骤与资料见[本日讲义](../docs/DAY16_SHARED_REDUCTION.md)。

以上是参考答案，不是个人学习结果。

## 验收记录

- [ ] 能解释原理
- [ ] 能自行复现
- [ ] 能解释一次失败
- [ ] 能说明当前实现不支持什么

个人笔记：

待填写。